# 🚀 AI-Based Insurance Policy & T&C Summarizer

## Project Features (As Per Report)
- **Domain-Adapted Summarization** using BART-Large-CNN
- **Insurance-Specific NER** for coverage, exclusions, monetary limits, time periods
- **Risk Scoring** (Critical/High/Medium/Low)
- **Clause-Level Traceability** linking summaries to source text
- **Multi-Format Input** (PDF, Images, DOCX, TXT, HTML)

---

## 📋 Step 1: Verify GPU Runtime
Go to **Runtime** → **Change runtime type** → Select **T4 GPU**

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → GPU")

## 📦 Step 2: Install Dependencies

In [ ]:
%%capture
!pip install transformers>=4.40.0 accelerate>=0.28.0 sentencepiece>=0.2.0
!pip install spacy>=3.7.0 pdfplumber>=0.11.0 pdf2image>=1.17.0
!pip install pytesseract>=0.3.10 python-docx>=1.1.0 beautifulsoup4>=4.12.0 lxml>=5.1.0
!pip install Pillow>=10.2.0 fastapi>=0.110.0 uvicorn>=0.27.0 pyngrok>=7.1.0 nest-asyncio>=1.6.0
!apt-get update -qq && apt-get install -qq -y tesseract-ocr poppler-utils
!python -m spacy download en_core_web_lg
print("✅ All dependencies installed!")

## 🤖 Step 3: Load GPU-Optimized Summarization Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "facebook/bart-large-cnn"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Loading model: {MODEL_NAME}")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32
).to(device)
model.eval()

print(f"✅ Model loaded on {device} ({model.num_parameters():,} parameters)")

In [ ]:
import spacy
nlp = spacy.load("en_core_web_lg")
print(f"✅ spaCy model loaded: en_core_web_lg")

## 📄 Step 4: Multi-Format Document Extractor with Clause Segmentation

In [ ]:
import os, re
from pathlib import Path
from io import BytesIO
from dataclasses import dataclass, field
from typing import List, Dict, Optional
import pdfplumber
import pytesseract
from PIL import Image
from pdf2image import convert_from_path, convert_from_bytes
from bs4 import BeautifulSoup
from docx import Document as DocxDocument

@dataclass
class Clause:
    clause_id: str
    text: str
    clause_type: str = "general"
    page_number: int = 1
    start_char: int = 0
    end_char: int = 0
    risk_level: str = "low"
    summary: str = ""
    entities: List[Dict] = field(default_factory=list)

@dataclass
class DocumentResult:
    full_text: str
    clauses: List[Clause]
    metadata: Dict
    summary: str = ""
    entities: List[Dict] = field(default_factory=list)

class MultiFormatExtractor:
    SUPPORTED_FORMATS = {
        'pdf': ['.pdf'], 'image': ['.jpg', '.jpeg', '.png', '.tiff', '.bmp', '.gif'],
        'docx': ['.docx'], 'text': ['.txt'], 'html': ['.html', '.htm']
    }
    
    CLAUSE_TYPE_KEYWORDS = {
        'coverage': ['cover', 'coverage', 'benefit', 'entitled', 'eligible', 'include'],
        'exclusion': ['exclud', 'not cover', 'exception', 'does not', 'shall not', 'limitation'],
        'claim': ['claim', 'submit', 'notify', 'within days', 'reimburs'],
        'payment': ['premium', 'payment', 'fee', 'charge', 'payable', 'cost'],
        'termination': ['terminat', 'cancel', 'end of policy', 'discontinu'],
        'renewal': ['renew', 'extension', 'continue'],
        'liability': ['liable', 'liability', 'indemnif', 'hold harmless'],
        'confidential': ['confidential', 'privacy', 'data protection'],
        'dispute': ['dispute', 'arbitrat', 'jurisdiction', 'governing law']
    }
    
    def extract(self, file_path=None, file_bytes=None, file_ext=None, raw_text=None) -> DocumentResult:
        if raw_text:
            return self._process_text(raw_text, "raw_text")
        
        if file_path:
            file_ext = Path(file_path).suffix.lower()
        file_ext = file_ext.lower() if file_ext else ''
        if not file_ext.startswith('.'):
            file_ext = '.' + file_ext
        
        if file_ext in self.SUPPORTED_FORMATS['pdf']:
            text, pages = self._extract_pdf(file_path, file_bytes)
        elif file_ext in self.SUPPORTED_FORMATS['image']:
            text, pages = self._extract_image(file_path, file_bytes)
        elif file_ext in self.SUPPORTED_FORMATS['docx']:
            text, pages = self._extract_docx(file_path, file_bytes)
        elif file_ext in self.SUPPORTED_FORMATS['text']:
            text, pages = self._extract_text_file(file_path, file_bytes)
        elif file_ext in self.SUPPORTED_FORMATS['html']:
            text, pages = self._extract_html(file_path, file_bytes)
        else:
            raise ValueError(f"Unsupported format: {file_ext}")
        
        return self._process_text(text, file_ext, pages)
    
    def _process_text(self, text: str, source_format: str, pages=None) -> DocumentResult:
        text = self._clean_text(text)
        clauses = self._segment_clauses(text)
        for clause in clauses:
            clause.clause_type = self._classify_clause(clause.text)
        return DocumentResult(
            full_text=text, clauses=clauses,
            metadata={'format': source_format, 'total_chars': len(text), 'clause_count': len(clauses), 'page_count': len(pages) if pages else 1}
        )
    
    def _segment_clauses(self, text: str) -> List[Clause]:
        clauses = []
        # Split on numbered headings (e.g., '1. COVERAGE') or double newlines
        paragraphs = re.split(r'\n\s*\n|(?=^\d+\.\s+[A-Z])', text, flags=re.MULTILINE)
        current_pos = 0
        merged = []
        buffer = ""
        # Merge short fragments into complete paragraphs (min 120 chars)
        for para in paragraphs:
            para = para.strip()
            if not para:
                continue
            if len(buffer) > 0 and len(buffer) < 120:
                buffer = buffer + " " + para
            elif len(para) < 120 and len(buffer) == 0:
                buffer = para
            else:
                if buffer:
                    merged.append(buffer)
                buffer = para
        if buffer:
            # If last buffer is too short, merge with previous
            if len(buffer) < 120 and merged:
                merged[-1] = merged[-1] + " " + buffer
            else:
                merged.append(buffer)
        # Create clause objects from merged paragraphs
        for i, para in enumerate(merged):
            para = para.strip()
            if len(para) > 50:
                start = text.find(para[:80], current_pos)
                if start < 0:
                    start = current_pos
                end = start + len(para)
                clauses.append(Clause(clause_id=f"C{i+1:03d}", text=para, start_char=max(0, start), end_char=end))
                current_pos = end
        return clauses if clauses else [Clause(clause_id="C001", text=text, end_char=len(text))]
    
    def _classify_clause(self, text: str) -> str:
        text_lower = text.lower()
        scores = {}
        for clause_type, keywords in self.CLAUSE_TYPE_KEYWORDS.items():
            score = sum(1 for kw in keywords if kw in text_lower)
            if score > 0: scores[clause_type] = score
        return max(scores, key=scores.get) if scores else "general"
    
    def _clean_text(self, text: str) -> str:
        if not text: return ""
        text = re.sub(r'\n{3,}', '\n\n', text)
        text = re.sub(r' {2,}', ' ', text)
        return text.strip()
    
    def _extract_pdf(self, file_path=None, file_bytes=None):
        pages_text = []
        pdf_source = file_path if file_path else BytesIO(file_bytes)
        with pdfplumber.open(pdf_source) as pdf:
            for page in pdf.pages:
                text = page.extract_text() or ""
                pages_text.append(self._clean_text(text))
        if not any(pages_text):
            print("📷 No text found, attempting OCR...")
            return self._ocr_pdf(file_path, file_bytes)
        return "\n\n".join(pages_text), pages_text
    
    def _ocr_pdf(self, file_path=None, file_bytes=None):
        pages_text = []
        images = convert_from_path(file_path) if file_path else convert_from_bytes(file_bytes)
        for image in images:
            pages_text.append(self._clean_text(pytesseract.image_to_string(image)))
        return "\n\n".join(pages_text), pages_text
    
    def _extract_image(self, file_path=None, file_bytes=None):
        image = Image.open(file_path if file_path else BytesIO(file_bytes))
        text = self._clean_text(pytesseract.image_to_string(image))
        return text, [text]
    
    def _extract_docx(self, file_path=None, file_bytes=None):
        doc = DocxDocument(file_path if file_path else BytesIO(file_bytes))
        text = self._clean_text("\n\n".join([p.text for p in doc.paragraphs if p.text.strip()]))
        return text, [text]
    
    def _extract_text_file(self, file_path=None, file_bytes=None):
        text = open(file_path, 'r', encoding='utf-8').read() if file_path else file_bytes.decode('utf-8')
        return self._clean_text(text), [text]
    
    def _extract_html(self, file_path=None, file_bytes=None):
        html = open(file_path, 'r', encoding='utf-8').read() if file_path else file_bytes.decode('utf-8')
        soup = BeautifulSoup(html, 'lxml')
        for tag in soup(['script', 'style', 'nav', 'footer', 'header']): tag.decompose()
        text = self._clean_text(soup.get_text(separator='\n'))
        return text, [text]

extractor = MultiFormatExtractor()
print("✅ Multi-Format Extractor with Clause Segmentation initialized!")


## 🏷️ Step 5: Insurance-Domain Named Entity Recognition (NER)

In [ ]:
from dataclasses import dataclass, asdict
from typing import List, Dict
import re

@dataclass
class InsuranceEntity:
    entity_type: str
    value: str
    start: int
    end: int
    confidence: float = 0.9
    context: str = ""

class InsuranceNERExtractor:
    ENTITY_PATTERNS = {
        'monetary_amount': [r'\$[\d,]+(?:\.\d{2})?(?:\s*(?:million|billion|k|K|M|B))?', r'(?:USD|EUR|GBP|INR|Rs\.?)\s*[\d,]+(?:\.\d{2})?'],
        'deductible': [r'deductible\s*(?:of|:)?\s*\$?[\d,]+', r'\$?[\d,]+\s*deductible'],
        'premium': [r'premium\s*(?:of|:)?\s*\$?[\d,]+', r'\$?[\d,]+\s*(?:monthly|annual|yearly)\s*premium'],
        'limit': [r'(?:limit|maximum|cap)\s*(?:of|:)?\s*\$?[\d,]+', r'up\s+to\s+\$?[\d,]+'],
        'co_insurance': [r'\d+(?:\.\d+)?\s*%\s*co-?insurance', r'co-?insurance\s*(?:of)?\s*\d+(?:\.\d+)?\s*%'],
        'waiting_period': [r'(?:waiting|grace)\s+period\s*(?:of)?\s*\d+\s*(?:day|week|month|year)s?'],
        'time_period': [r'within\s+\d+\s*(?:day|week|month|year)s?', r'\d+\s*(?:day|week|month|year)s?\s*(?:of|from|after|before)'],
        'date': [r'(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},?\s+\d{4}'],
        'percentage': [r'\d+(?:\.\d+)?\s*%'],
        'coverage_item': [r'(?:covers?|coverage\s+(?:for|of|includes?))\s+([^,.;]{5,60})'],
        'exclusion': [r'(?:exclud(?:es?|ing|ed)|not\s+cover(?:ed)?|does\s+not\s+(?:cover|apply|include))\s+([^,.;]{5,80})'],
        'condition': [r'(?:provided\s+that|subject\s+to|on\s+condition\s+that)\s+([^,.;]{5,80})'],
        'obligation': [r'(?:must|shall|required\s+to|obligat(?:ed|ion))\s+([^,.;]{5,60})'],
    }

    RISK_KEYWORDS = {
        'critical': ['not covered', 'excluded', 'denied', 'void', 'terminate', 'forfeit', 'waive', 'lifetime limit'],
        'high': ['limitation', 'restriction', 'waiting period', 'pre-existing', 'deductible', 'maximum'],
        'medium': ['condition', 'provided that', 'subject to', 'may', 'reasonable'],
    }

    # ──── NEW: Human-readable explanation for every risk keyword ────
    RISK_EXPLANATIONS = {
        'not covered': 'This clause indicates certain items or services are NOT covered, which could lead to claim denial.',
        'excluded': 'This clause explicitly excludes specific scenarios from coverage. Review carefully to understand what is not protected.',
        'denied': 'This clause describes conditions under which a claim may be denied outright.',
        'void': 'This clause can render the entire policy void under certain conditions.',
        'terminate': 'This clause allows termination of the policy, potentially leaving the policyholder without coverage.',
        'forfeit': 'This clause may cause the policyholder to forfeit benefits or premiums already paid.',
        'waive': 'This clause involves waiving certain rights or protections.',
        'lifetime limit': 'This clause imposes a maximum lifetime payout, after which no further claims are honored.',
        'limitation': 'This clause restricts the scope or amount of coverage or benefits available.',
        'restriction': 'This clause places restrictions on when or how the coverage applies.',
        'waiting period': 'A waiting period delays coverage activation, meaning any claims during this time will NOT be paid.',
        'pre-existing': 'Pre-existing condition clauses can exclude coverage for previously diagnosed health issues.',
        'deductible': 'A deductible requires the policyholder to pay a portion out-of-pocket before insurance kicks in.',
        'maximum': 'This clause sets a maximum cap on the payable benefits.',
        'condition': 'This clause sets conditional requirements that must be met for coverage to apply.',
        'provided that': 'Coverage under this clause is conditional and applies only if specific criteria are met.',
        'subject to': 'Benefits under this clause are subject to additional terms or approvals.',
        'may': 'The use of "may" introduces discretionary power, meaning the insurer is not obligated to act.',
        'reasonable': 'The term "reasonable" is subjective and could be interpreted differently by the insurer vs. policyholder.',
    }

    def __init__(self, spacy_model):
        self.nlp = spacy_model
        self.patterns = {k: [re.compile(p, re.IGNORECASE) for p in v] for k, v in self.ENTITY_PATTERNS.items()}

    def extract(self, text: str) -> List[InsuranceEntity]:
        entities = []
        doc = self.nlp(text[:100000])
        spacy_map = {'MONEY': 'monetary_amount', 'PERCENT': 'percentage', 'DATE': 'date', 'ORG': 'party', 'PERSON': 'party'}
        for ent in doc.ents:
            if ent.label_ in spacy_map:
                entities.append(InsuranceEntity(entity_type=spacy_map[ent.label_], value=ent.text.strip(), start=ent.start_char, end=ent.end_char, confidence=0.85, context=text[max(0, ent.start_char-30):min(len(text), ent.end_char+30)]))
        for entity_type, patterns in self.patterns.items():
            for pattern in patterns:
                for match in pattern.finditer(text):
                    value = match.group(1) if match.lastindex else match.group(0)
                    if len(value.strip()) > 2:
                        entities.append(InsuranceEntity(entity_type=entity_type, value=value.strip()[:100], start=match.start(), end=match.end(), confidence=0.90, context=text[max(0, match.start()-30):min(len(text), match.end()+30)]))
        seen = set()
        unique = []
        for e in entities:
            key = (e.entity_type, e.value.lower())
            if key not in seen:
                seen.add(key)
                unique.append(e)
        return unique

    # ──── ORIGINAL: Simple risk label (kept for backward compatibility) ────
    def assess_risk(self, text: str) -> str:
        text_lower = text.lower()
        for level, keywords in self.RISK_KEYWORDS.items():
            if any(kw in text_lower for kw in keywords): return level
        return 'low'

    # ──── NEW: Detailed risk assessment with explanations ────
    def assess_risk_detailed(self, text: str) -> dict:
        text_lower = text.lower()
        risk_level = 'low'
        matched_keywords = []
        explanations = []

        for level, keywords in self.RISK_KEYWORDS.items():
            for kw in keywords:
                if kw in text_lower:
                    # Upgrade risk level to the highest found
                    if risk_level == 'low' or \
                       (level == 'critical' and risk_level != 'critical') or \
                       (level == 'high' and risk_level in ('medium', 'low')):
                        risk_level = level
                    matched_keywords.append(kw)
                    explanations.append(self.RISK_EXPLANATIONS.get(kw, f'Keyword "{kw}" indicates potential risk.'))

        return {
            'level': risk_level,
            'matched_keywords': matched_keywords,
            'explanations': explanations,
            'keyword_count': len(matched_keywords)
        }

ner_extractor = InsuranceNERExtractor(nlp)
print("✅ Insurance-Domain NER Extractor initialized (with detailed risk analysis)!")
print(f"   Entity types: {list(ner_extractor.ENTITY_PATTERNS.keys())}")

## 🧠 Step 6: GPU Summarizer with Clause-Level Traceability

In [ ]:
from dataclasses import asdict
from typing import List, Dict
import time

class GPUSummarizer:
    def __init__(self, model_instance, tokenizer_instance, device_instance, ner_extractor):
        self.model = model_instance
        self.tokenizer = tokenizer_instance
        self.device = device_instance
        self.ner = ner_extractor
        self.model_name = model_instance.config._name_or_path

    def summarize_document(self, doc_result, max_length: int = 250, min_length: int = 50):
        doc_result.summary = self._generate_summary(doc_result.full_text, max_length, min_length)
        doc_result.entities = [asdict(e) for e in self.ner.extract(doc_result.full_text)]
        for clause in doc_result.clauses:
            # ── CHANGED: max_length 100 → 200, min_length 20 → 40 for longer summaries ──
            clause.summary = self._generate_summary(clause.text, max_length=200, min_length=40) if len(clause.text) > 100 else clause.text
            clause.risk_level = self.ner.assess_risk(clause.text)
            # ── NEW: Store detailed risk analysis with explanations ──
            clause.risk_details = self.ner.assess_risk_detailed(clause.text)
            clause.entities = [asdict(e) for e in self.ner.extract(clause.text)]
        return doc_result

    def _generate_summary(self, text: str, max_length: int = 150, min_length: int = 40) -> str:
        if not text or len(text.strip()) < 50: return text.strip() if text else ""
        text = text[:4096] if len(text) > 4096 else text
        try:
            inputs = self.tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(self.device)
            with torch.no_grad():
                summary_ids = self.model.generate(inputs["input_ids"], attention_mask=inputs["attention_mask"], max_length=max_length, min_length=min_length, num_beams=4, early_stopping=True, do_sample=False)
            return self.tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        except Exception as e:
            print(f"Summarization error: {e}")
            return text[:300] + "..."

    def get_summary_with_traceability(self, doc_result) -> Dict:
        return {
            "overall_summary": doc_result.summary,
            "metadata": doc_result.metadata,
            "entities_summary": self._group_entities(doc_result.entities),
            # ── CHANGED: source text no longer truncated to 200 chars; entities limit raised to 10 ──
            "clauses": [{
                "clause_id": c.clause_id,
                "clause_type": c.clause_type,
                "risk_level": c.risk_level,
                "risk_details": getattr(c, 'risk_details', {}),
                "summary": c.summary,
                "source_text": c.text,
                "source_excerpt": c.text[:500] + "..." if len(c.text) > 500 else c.text,
                "entities": c.entities[:10],
                "char_range": [c.start_char, c.end_char]
            } for c in doc_result.clauses],
            "risk_summary": self._get_risk_summary(doc_result.clauses)
        }

    def _group_entities(self, entities):
        grouped = {}
        for e in entities:
            t = e['entity_type']
            if t not in grouped: grouped[t] = []
            if len(grouped[t]) < 10: grouped[t].append(e['value'])  # ── CHANGED: limit raised 5 → 10 ──
        return grouped

    def _get_risk_summary(self, clauses):
        counts = {'critical': 0, 'high': 0, 'medium': 0, 'low': 0}
        for c in clauses: counts[c.risk_level] = counts.get(c.risk_level, 0) + 1
        return counts

gpu_summarizer = GPUSummarizer(model, tokenizer, device, ner_extractor)
print("✅ GPU Summarizer with Detailed Traceability initialized!")

This step is used to help the remaining cells work properly. Stop using it if not working correctly.

In [ ]:
def print_detailed_analysis(analysis, elapsed=None):
    """Reusable function to print full detailed analysis output."""

    if elapsed is not None:
        print(f"\n⏱️ Processing Time: {elapsed:.2f}s\n")

    # ── Overall Summary ──
    print("📝 OVERALL SUMMARY:")
    print("-" * 60)
    print(analysis['overall_summary'])
    print("-" * 60)

    # ── Metadata ──
    print(f"\n📊 DOCUMENT METADATA:")
    print(f"   Clauses: {analysis['metadata']['clause_count']}")
    print(f"   Characters: {analysis['metadata']['total_chars']:,}")

    # ── Risk Summary ──
    print(f"\n⚠️ RISK SUMMARY:")
    for level, count in analysis['risk_summary'].items():
        emoji = {'critical': '🔴', 'high': '🟠', 'medium': '🟡', 'low': '🟢'}[level]
        print(f"   {emoji} {level.upper()}: {count} clauses")

    # ── All Entities ──
    print(f"\n🏷️ KEY ENTITIES:")
    for entity_type, values in analysis['entities_summary'].items():
        print(f"   📌 {entity_type.upper()}: {', '.join(values)}")

    # ── Detailed Clause-Level Analysis with Full Traceability ──
    print(f"\n{'='*60}")
    print(f"📋 DETAILED CLAUSE-LEVEL ANALYSIS")
    print(f"{'='*60}")

    for clause in analysis['clauses']:
        risk_emoji = {'critical': '🔴', 'high': '🟠', 'medium': '🟡', 'low': '🟢'}[clause['risk_level']]

        print(f"\n   {'─'*56}")
        print(f"   {risk_emoji} [{clause['clause_id']}] Type: {clause['clause_type'].upper()} | Risk: {clause['risk_level'].upper()}")
        print(f"   {'─'*56}")

        # Full Summary
        print(f"\n   📝 Summary:")
        print(f"      {clause['summary']}")

        # Full Source Text (Traceability)
        print(f"\n   📖 Source Text (Traceability):")
        print(f"      \"{clause['source_excerpt']}\"")
        print(f"      [Character range: {clause['char_range'][0]} - {clause['char_range'][1]}]")

        # Risk Explanation
        risk_details = clause.get('risk_details', {})
        if risk_details and risk_details.get('matched_keywords'):
            print(f"\n   ⚠️ Risk Analysis ({risk_details['keyword_count']} risk indicator(s) found):")
            print(f"      Triggered Keywords: {', '.join(risk_details['matched_keywords'])}")
            for i, exp in enumerate(risk_details['explanations'], 1):
                print(f"      {i}. {exp}")
        else:
            print(f"\n   ✅ Risk Analysis: No high-risk keywords detected in this clause.")

        # All Entities
        if clause['entities']:
            print(f"\n   🏷️ Entities Found ({len(clause['entities'])}):")
            for e in clause['entities']:
                print(f"      • {e['entity_type'].upper()}: {e['value']}  (confidence: {e.get('confidence', 'N/A')})")

        print()

    # ── High-Risk Clauses Highlight ──
    high_risk = [c for c in analysis['clauses'] if c['risk_level'] in ['critical', 'high']]
    if high_risk:
        print(f"{'='*60}")
        print(f"🚨 HIGH-RISK CLAUSES SUMMARY ({len(high_risk)} found)")
        print(f"{'='*60}")
        for clause in high_risk:
            risk_emoji = {'critical': '🔴', 'high': '🟠'}[clause['risk_level']]
            print(f"\n   {risk_emoji} [{clause['clause_id']}] {clause['clause_type'].upper()} — Risk: {clause['risk_level'].upper()}")
            print(f"      Summary: {clause['summary']}")
            if clause.get('risk_details') and clause['risk_details'].get('explanations'):
                for exp in clause['risk_details']['explanations']:
                    print(f"      → {exp}")

print("✅ print_detailed_analysis() helper function defined!")

## 🧪 Step 7: Test with Sample Insurance Policy

In [ ]:
import time
import json

sample_policy = """
HEALTH INSURANCE POLICY - TERMS AND CONDITIONS

1. COVERAGE
This policy covers hospitalization expenses up to $500,000 annually. The policyholder is entitled 
to coverage for all medically necessary treatments, including surgery, diagnostic tests, and 
prescription medications. Coverage includes emergency room visits with a $100 co-pay.

2. PREMIUM AND PAYMENT
The annual premium is $2,400 payable monthly ($200/month) or annually with 5% discount.
Payment must be received within 15 days of the due date. Late payments will incur a 2% penalty.

3. EXCLUSIONS
The following are explicitly excluded from coverage:
- Cosmetic procedures and elective surgery not medically necessary
- Experimental treatments and clinical trials
- Self-inflicted injuries
- Pre-existing conditions diagnosed within 12 months prior to policy start date
- Injuries resulting from participation in extreme sports

4. WAITING PERIOD
A 30-day waiting period applies for all non-emergency treatments. For maternity coverage,
a 9-month waiting period applies from the policy effective date.

5. CLAIMS PROCESS
All claims must be submitted within 90 days of treatment. A deductible of $1,000 applies per 
claim year. The maximum out-of-pocket limit is $10,000 per year. Claims exceeding $5,000
require pre-authorization.

6. TERMINATION
Either party may terminate this policy with 30 days written notice. The insurer may terminate
immediately for fraud or material misrepresentation. No refunds for partial months.

This policy is issued by ABC Insurance Company, effective from January 1, 2024.
Policy Number: HI-2024-001234
"""

print("📄 Processing Sample Insurance Policy...")
print("=" * 60)

start = time.time()
doc_result = extractor.extract(raw_text=sample_policy)
doc_result = gpu_summarizer.summarize_document(doc_result)
analysis = gpu_summarizer.get_summary_with_traceability(doc_result)
elapsed = time.time() - start

print(f"\n⏱️ Processing Time: {elapsed:.2f}s\n")

# ── Overall Summary ──
print("📝 OVERALL SUMMARY:")
print("-" * 60)
print(analysis['overall_summary'])
print("-" * 60)

# ── Metadata ──
print(f"\n📊 DOCUMENT METADATA:")
print(f"   Clauses: {analysis['metadata']['clause_count']}")
print(f"   Characters: {analysis['metadata']['total_chars']:,}")

# ── Risk Summary ──
print(f"\n⚠️ RISK SUMMARY:")
for level, count in analysis['risk_summary'].items():
    emoji = {'critical': '🔴', 'high': '🟠', 'medium': '🟡', 'low': '🟢'}[level]
    print(f"   {emoji} {level.upper()}: {count} clauses")

# ── All Entities ──
print(f"\n🏷️ KEY ENTITIES:")
for entity_type, values in analysis['entities_summary'].items():
    print(f"   📌 {entity_type.upper()}: {', '.join(values)}")

# ── Detailed Clause-Level Analysis with Full Traceability ──
print(f"\n{'='*60}")
print(f"📋 DETAILED CLAUSE-LEVEL ANALYSIS")
print(f"{'='*60}")

for clause in analysis['clauses']:
    risk_emoji = {'critical': '🔴', 'high': '🟠', 'medium': '🟡', 'low': '🟢'}[clause['risk_level']]

    print(f"\n   {'─'*56}")
    print(f"   {risk_emoji} [{clause['clause_id']}] Type: {clause['clause_type'].upper()} | Risk: {clause['risk_level'].upper()}")
    print(f"   {'─'*56}")

    # Full Summary
    print(f"\n   📝 Summary:")
    print(f"      {clause['summary']}")

    # Full Source Text (Traceability)
    print(f"\n   📖 Source Text (Traceability):")
    print(f"      \"{clause['source_excerpt']}\"")
    print(f"      [Character range: {clause['char_range'][0]} - {clause['char_range'][1]}]")

    # Risk Explanation
    risk_details = clause.get('risk_details', {})
    if risk_details and risk_details.get('matched_keywords'):
        print(f"\n   ⚠️ Risk Analysis ({risk_details['keyword_count']} risk indicator(s) found):")
        print(f"      Triggered Keywords: {', '.join(risk_details['matched_keywords'])}")
        for i, exp in enumerate(risk_details['explanations'], 1):
            print(f"      {i}. {exp}")
    else:
        print(f"\n   ✅ Risk Analysis: No high-risk keywords detected in this clause.")

    # All Entities
    if clause['entities']:
        print(f"\n   🏷️ Entities Found ({len(clause['entities'])}):")
        for e in clause['entities']:
            print(f"      • {e['entity_type'].upper()}: {e['value']}  (confidence: {e.get('confidence', 'N/A')})")

    print()

## 📤 Step 8: Process Your Documents

### How to upload files (Reliable Method):
1. Click the **📁 folder icon** in the LEFT SIDEBAR
2. Click the **↑ upload button** at the top of the file panel
3. Select your file and wait for it to upload
4. Copy the filename and paste it in the cell below
5. Run the cell

In [ ]:
import time
import os
from pathlib import Path

# ============================================================
# 📁 CHANGE THIS TO YOUR UPLOADED FILENAME
# ============================================================
FILENAME = "your_file.pdf"  # ← Change this!
# ============================================================

# Check /content folder for uploaded files
print("📂 Files currently in /content:")
files_found = [f for f in os.listdir('/content') if not f.startswith('.') and os.path.isfile(f'/content/{f}')]
for f in files_found[:10]:
    size = os.path.getsize(f'/content/{f}')
    print(f"   📄 {f} ({size:,} bytes)")

if not files_found:
    print("   ⚠️ No files found. Upload using the sidebar (folder icon on left).")

# Process if file exists
filepath = f"/content/{FILENAME}"
if os.path.exists(filepath):
    print(f"\n{'='*60}")
    print(f"📄 Processing: {FILENAME}")
    print(f"{'='*60}")

    start = time.time()
    doc_result = extractor.extract(file_path=filepath)
    doc_result = gpu_summarizer.summarize_document(doc_result)
    analysis = gpu_summarizer.get_summary_with_traceability(doc_result)
    elapsed = time.time() - start

    print_detailed_analysis(analysis, elapsed)

else:
    if FILENAME != "your_file.pdf":
        print(f"\n❌ File not found: {filepath}")
    print("\n📋 Instructions:")
    print("   1. Click the 📁 folder icon in the left sidebar")
    print("   2. Click ↑ upload button at top of file panel")
    print("   3. Select your policy document")
    print("   4. Change FILENAME above to match your uploaded file")
    print("   5. Run this cell again")

## ✏️ Step 9: Paste Text Directly

In [ ]:
# Paste your policy text here
policy_text = """
PASTE YOUR POLICY OR TERMS & CONDITIONS TEXT HERE
"""

if "PASTE YOUR" not in policy_text:
    print("📄 Processing pasted text...")
    print("=" * 60)

    start = time.time()
    doc_result = extractor.extract(raw_text=policy_text)
    doc_result = gpu_summarizer.summarize_document(doc_result)
    analysis = gpu_summarizer.get_summary_with_traceability(doc_result)
    elapsed = time.time() - start

    print_detailed_analysis(analysis, elapsed)

else:
    print("⚠️ Replace the placeholder text above with your policy/T&C and run again.")

## 🌐 Step 10: FastAPI Server (For Backend Integration)

In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException, Form
from pydantic import BaseModel
from typing import List, Dict, Optional
import nest_asyncio
import uvicorn
import json

nest_asyncio.apply()

app = FastAPI(title="Insurance Policy Summarizer API", version="2.0")

class TextRequest(BaseModel):
    text: str
    max_length: int = 250
    min_length: int = 50

class FeedbackRequest(BaseModel):
    document_id: str
    rating: int
    corrections: Optional[str] = None
    comments: Optional[str] = None

@app.get("/")
def health():
    return {"status": "healthy", "gpu": torch.cuda.is_available(), "model": MODEL_NAME, "version": "2.0"}

@app.post("/summarize")
def summarize_text(request: TextRequest):
    try:
        doc_result = extractor.extract(raw_text=request.text)
        doc_result = gpu_summarizer.summarize_document(doc_result, request.max_length, request.min_length)
        return gpu_summarizer.get_summary_with_traceability(doc_result)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/upload")
async def process_file(file: UploadFile = File(...)):
    content = await file.read()
    ext = Path(file.filename).suffix.lower()
    try:
        doc_result = extractor.extract(file_bytes=content, file_ext=ext)
        doc_result = gpu_summarizer.summarize_document(doc_result)
        return gpu_summarizer.get_summary_with_traceability(doc_result)
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.post("/feedback")
def submit_feedback(feedback: FeedbackRequest):
    print(f"Feedback received: {feedback.rating}/5 for {feedback.document_id}")
    return {"status": "received", "message": "Thank you for your feedback!"}

print("✅ FastAPI application ready!")
print("   Endpoints: /, /summarize, /upload, /feedback")

In [ ]:
# Set your ngrok auth token
NGROK_AUTH_TOKEN = "39RFdDXP4uvaQJVtWqUyu5xlW8m_3dith7xoAnD7iXyfF1QyQ"  # Get from https://ngrok.com/

import threading

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8080)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("🚀 Server running on port 8080")

if NGROK_AUTH_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8080)
    print(f"\n🌐 Public URL: {public_url}")
    print(f"📚 API Docs: {public_url}/docs")
else:
    print("\n⚠️ Add NGROK_AUTH_TOKEN to expose API externally")
    print("   Local access: http://localhost:8080/docs")

---
## ✅ System Complete!

### Features Implemented:
| Feature | Status |
|---------|--------|
| Multi-Format Input | ✅ PDF, Images (OCR), DOCX, TXT, HTML |
| Summarization | ✅ BART-Large-CNN on GPU |
| Insurance NER | ✅ Coverage, Exclusions, Monetary, Temporal |
| Risk Scoring | ✅ Critical/High/Medium/Low |
| Traceability | ✅ Summary → Source clause linking |
| FastAPI Backend | ✅ /summarize, /upload, /feedback |